In [8]:
import pandas as pd
import plotly.express as px
from movie_analytics.database import get_engine

In [9]:
engine = get_engine()
movies_df = pd.read_sql("SELECT * FROM movies_full_dataset",engine)
movies_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6431 entries, 0 to 6430
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   imdb_id     6431 non-null   str    
 1   title       6431 non-null   str    
 2   year        6431 non-null   int64  
 3   genres      6429 non-null   str    
 4   avg_rating  6431 non-null   float64
 5   votes       6431 non-null   int64  
 6   popularity  6431 non-null   float64
 7   revenue     6431 non-null   int64  
 8   budget      6431 non-null   int64  
dtypes: float64(2), int64(4), str(3)
memory usage: 452.3 KB


In [10]:
df_cleaned = movies_df.dropna(subset=["genres"]).copy()
df_cleaned["genres_list"] = df_cleaned["genres"].str.split(',')
df_cleaned["genres_weight"] = 1/df_cleaned["genres_list"].apply(len)
df_exploded = df_cleaned.explode("genres_list")
df = df_exploded.rename(columns={"genres_list":"genre","genres_weight":"genre_weight"})
df


,imdb_id,title,year,genres,avg_rating,votes,popularity,revenue,budget,genre,genre_weight
0,tt0468569,The Dark Knight,2008,"Action,Crime,Drama",9.1,3162226,130.6430,1004558444,185000000,Action,0.333333
0,tt0468569,The Dark Knight,2008,"Action,Crime,Drama",9.1,3162226,130.6430,1004558444,185000000,Crime,0.333333
0,tt0468569,The Dark Knight,2008,"Action,Crime,Drama",9.1,3162226,130.6430,1004558444,185000000,Drama,0.333333
1,tt1375666,Inception,2010,"Action,Adventure,Sci-Fi",8.8,2811917,83.9520,825532764,160000000,Action,0.333333
1,tt1375666,Inception,2010,"Action,Adventure,Sci-Fi",8.8,2811917,83.9520,825532764,160000000,Adventure,0.333333
...,...,...,...,...,...,...,...,...,...,...,...
6426,tt23723194,The 166 Minutes of Madness,2022,Horror,7.8,7,0.6000,100,5000,Horror,1.000000
6427,tt13129744,"Estremecer, Lejos de ti.",2018,Thriller,9.0,7,0.6000,10000,10000,Thriller,1.000000
6428,tt36955396,Mamá quiero ser futbolista profesional,2022,Documentary,9.7,7,0.0357,3000,15000,Documentary,1.000000
6429,tt9536384,Mai Mai Ti's 2008,2008,Drama,6.8,7,0.6000,20707,414147,Drama,1.000000


In [11]:
genres_list = ["Action", "Adventure", "Animation", "Comedy", "Drama", "Fantasy", "Horror", "Mystery", "Romance", "Sci-Fi", "Thriller"]

colors = [
    "#636EFA", # Royal Blue
    "#EF553B", # Sunset Red
    "#00CC96", # Emerald Green
    "#AB63FA", # Vivid Purple
    "#FFA15A", # Safety Orange
    "#19D3F3", # Sky Blue
    "#FF6692", # Hot Pink
    "#B6E880", # Chartreuse
    "#FFD700", # Gold
    "#72B7B2", # Sage/Grey-Teal
    "#542788"  # Deep Indigo
]



In [12]:
df_ratings = df.copy()
df_ratings = df_ratings[df_ratings["genre"].isin(genres_list)]
df_ratings = df_ratings.groupby("genre", as_index=False).agg(mean_rating=("avg_rating","mean"))[["genre","mean_rating"]]

fig=px.bar(df_ratings,
        x="genre", 
        y="mean_rating",
        color="genre",
        color_discrete_sequence=colors
    )
fig.update_yaxes(range=[5,7])

fig.show()




In [13]:
df_revenue_share = df.copy()
df_revenue_share = df_revenue_share[df_revenue_share["genre"].isin(genres_list)]
df_revenue_share["weighted_revenue"] = df_revenue_share["revenue"]*df_revenue_share["genre_weight"]
df_revenue_share = df_revenue_share.groupby("genre",as_index=False).agg(sum_weighted_revenue=("weighted_revenue","sum"))[["genre","sum_weighted_revenue"]]
df_revenue_share["weighted_revenue_share"] = (df_revenue_share["sum_weighted_revenue"])*100/(df_revenue_share["sum_weighted_revenue"].sum())

fig = px.bar(
    df_revenue_share,
    x="genre",
    y="weighted_revenue_share",
    color="genre",
    color_discrete_sequence=colors
)

fig.show()

In [14]:
df_revenue_share = df.copy()
df_revenue_share = df_revenue_share[df_revenue_share["genre"].isin(genres_list)]
df_revenue_share["weighted_revenue"] = df_revenue_share["revenue"]*df_revenue_share["genre_weight"]
df_revenue_share = df_revenue_share.groupby(["year","genre"],as_index=False).agg(sum_weighted_revenue=("weighted_revenue","sum"))[["year","genre","sum_weighted_revenue"]]
df_revenue_share["weighted_revenue_share"] = (df_revenue_share["sum_weighted_revenue"])*100/(df_revenue_share.groupby("year")["sum_weighted_revenue"].transform(sum))

fig = px.line(
    df_revenue_share,
    x="year",
    y="weighted_revenue_share",
    color="genre",
    color_discrete_sequence=colors
)

fig.update_layout(
    legend_itemclick="toggleothers", 
    legend_itemdoubleclick="toggle"      
)

fig.show()